In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "Calculator/bin/Debug/net9.0/Calculator.dll"
open System
open System.IO
open LectureCalculator

# Parsing with an F# Visitor

This notebook parses and evaluates the same assignment language. FsLexYacc replaces ANTLR generation; an explicit typed AST replaces dynamically patched visitor methods.

The FsYacc grammar is in `Calculator/Parser.fsy`. The corresponding FsLex scanner is in `Calculator/Lexer.fsl`.

In [ ]:
File.ReadAllText("Calculator/Parser.fsy") |> printfn "%s"

The grammar builds a typed tree without evaluating it. Parsing and visiting remain separate steps; F# pattern matching dispatches to the small visitor functions below.

In [ ]:
// Build Calculator/Calculator.fsproj; it generates FsLex/FsYacc F# files.

The project builds `Parser.fs` and `Lexer.fs` from the specification. `Ast.fs` defines the expression and statement types returned by the generated parser.

In [ ]:
File.ReadAllText("Calculator/Ast.fs") |> printfn "%s"

The visitor state is a dictionary of arbitrary-precision integers, retaining Python integer semantics. Undefined identifiers evaluate to zero. Each visitor function implements one operation; assignment and print receive the expression evaluator explicitly.

In [ ]:
let values = Collections.Generic.Dictionary<string,bigint>()

In [ ]:
let visitAssign evaluate (name,expression) =
    let value = evaluate expression
    values.[name] <- value
    value

In [ ]:
let visitPrint evaluate expression =
    let value = evaluate expression
    printfn "%A" value
    value

In [ ]:
let visitAdd (lhs: bigint) (rhs: bigint) = lhs + rhs

In [ ]:
let visitSub (lhs: bigint) (rhs: bigint) = lhs - rhs

In [ ]:
let visitMul (lhs: bigint) (rhs: bigint) = lhs * rhs

In [ ]:
let visitDiv (lhs: bigint) (rhs: bigint) =
    if rhs = 0I then raise (DivideByZeroException())
    let quotient, remainder = lhs / rhs, lhs % rhs
    // Python // rounds down; bigint division itself truncates toward zero.
    if remainder <> 0I && ((lhs < 0I) <> (rhs < 0I)) then quotient - 1I else quotient

In [ ]:
let visitParens value = value // Parentheses affect parsing, not the AST value.

In [ ]:
let visitInt (number: bigint) = number

In [ ]:
let visitId name =
    match values.TryGetValue(name) with
    | true, value -> value
    | false, _ -> 0I

In [ ]:
let data = """
a = 3;
b = 4;
x = a * a + b * b;
c = 5;
print(c * c);
print(x);
"""

`parseData source` starts with an empty environment, parses the statement list, and visits statements in source order. Arithmetic precedence remains in the grammar; parentheses do not need a separate AST node.

In [ ]:
let rec evaluate expression =
    match expression with
    | Integer number -> visitInt number
    | Identifier name -> visitId name
    | Binary(op,lhs,rhs) ->
        let left,right = evaluate lhs,evaluate rhs
        match op with
        | "+" -> visitAdd left right
        | "-" -> visitSub left right
        | "*" -> visitMul left right
        | "/" -> visitDiv left right
        | _ -> failwithf "Unknown operator %s" op
let parseData source =
    values.Clear()
    let statements = Calculator.parse source
    statements |> List.map (function
        | Assignment(name,expression) -> visitAssign evaluate (name,expression)
        | Print expression -> visitPrint evaluate expression)

In [ ]:
let result = parseData data
checkNotebook (result |> List.rev |> List.take 2 = [25I;25I])
checkNotebook (visitDiv -3I 2I = -2I)
checkNotebook (visitDiv 3I -2I = -2I)
checkNotebook (visitId "undefined" = 0I)

In [ ]:
// Generated files stay in the ignored build directory; no notebook cleanup is needed.